In [ ]:
import warnings
warnings.filterwarnings('ignore') # Suppress warnings, especially from transformers

# Install necessary libraries
%pip install transformers torch gradio --quiet

# Import libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import gradio as gr

# Load a pre-trained model and tokenizer from Hugging Face
# Using 'microsoft/DialoGPT-small' for a general-purpose conversational chatbot
model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Define the chatbot's response generation function
def predict(message, history):
    # Gradio's history is a list of lists: [[user1, bot1], [user2, bot2], ...]
    # We need to format this into a single string for the DialoGPT model,
    # including the end-of-sentence token for proper context.
    full_conversation = ""
    for human_utterance, assistant_utterance in history:
        full_conversation += human_utterance + tokenizer.eos_token
        full_conversation += assistant_utterance + tokenizer.eos_token
    # Add the current user's message to the conversation
    full_conversation += message + tokenizer.eos_token

    # Encode the full conversation
    input_ids = tokenizer.encode(full_conversation, return_tensors='pt')

    # Generate a response from the model
    # Use sampling for more diverse and conversational responses
    output = model.generate(
        input_ids,
        max_length=50, # Maximum length of the generated response - reduced for optimization
        min_length=10,   # Ensure at least 10 tokens are generated - reduced for optimization
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True, # Enable sampling
        top_k=50,       # Consider top 50 most likely words
        top_p=0.95,     # Nucleus sampling: consider words whose cumulative probability exceeds 0.95
        temperature=0.7, # Control randomness; lower temp means less random
        num_return_sequences=1, # Generate only one response
        no_repeat_ngram_size=3 # Prevent repetitive phrases
    )

    # Decode only the newly generated part of the conversation (the bot's response)
    response = tokenizer.decode(output[:, input_ids.shape[-1]:][0], skip_special_tokens=True)

    # Fallback for empty responses
    if not response.strip():
        return "Promiňte, ale nemohu teď odpovědět. Zkuste to prosím znovu."
    return response

# Create and launch the Gradio ChatInterface
# This will provide an an interactive chat widget directly in the notebook.
gr.ChatInterface(
    predict,
    chatbot=gr.Chatbot(height=300), # Display area for chat history
    textbox=gr.Textbox(placeholder="Zeptejte se mě na něco... Zadejte svou otázku zde.", container=False, scale=7), # Input box with more explicit instruction
    title="Baseline Chatbot (DialoGPT)", # Title of the interface
    description="Jednoduchý konverzační AI chatbot používající model DialoGPT-small z Hugging Face pro účely cvičení a kurzu.", # Description
    examples=[["Ahoj"], ["Jak se máš?"], ["Co je tvůj účel?"]] # Example questions
).launch(inline=True, share=False) # Launch the interface inline in the notebook; do not share publicly

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

It seems the connection issues persist. To further reduce memory consumption and improve stability, we can try **8-bit model quantization**. This technique loads the model using less memory by representing its weights with fewer bits.

First, we need to install the `bitsandbytes` and `accelerate` libraries, which enable this functionality.

In [ ]:
# Install libraries for 8-bit quantization
%pip install bitsandbytes accelerate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 16.6 MB/s eta 0:00:00


Now, I will modify the model loading code to enable 8-bit quantization. This should significantly reduce the memory footprint of the DialoGPT-small model.

In [ ]:
import warnings
warnings.filterwarnings('ignore') # Suppress warnings, especially from transformers

# Import libraries
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import gradio as gr

# Load a pre-trained model and tokenizer from Hugging Face
model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define quantization configuration for 8-bit loading
# This is the correct way to pass quantization parameters for AutoModels
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

# Load the model with 8-bit quantization to reduce memory usage
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quantization_config)

# Define the chatbot's response generation function
def predict(message, history):
    full_conversation = ""
    for human_utterance, assistant_utterance in history:
        full_conversation += human_utterance + tokenizer.eos_token
        full_conversation += assistant_utterance + tokenizer.eos_token
    full_conversation += message + tokenizer.eos_token

    input_ids = tokenizer.encode(full_conversation, return_tensors='pt')

    # Generate a response from the model
    output = model.generate(
        input_ids.to(model.device), # Ensure input_ids are on the correct device for 8-bit model
        max_length=50,
        min_length=10,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        num_return_sequences=1,
        no_repeat_ngram_size=3
    )

    response = tokenizer.decode(output[:, input_ids.shape[-1]:][0], skip_special_tokens=True)

    if not response.strip():
        return "Promiňte, ale nemohu teď odpovědět. Zkuste to prosím znovu."
    return response

# Create and launch the Gradio ChatInterface
gr.ChatInterface(
    predict,
    chatbot=gr.Chatbot(height=300),
    textbox=gr.Textbox(placeholder="Zeptejte se mě na něco... Zadejte svou otázku zde.", container=False, scale=7),
    title="Baseline Chatbot (DialoGPT) - 8-bit Quantized",
    description="Jednoduchý konverzační AI chatbot používající 8-bit kvantizovaný model DialoGPT-small z Hugging Face pro účely cvičení a kurzu.",
    examples=[["Ahoj"], ["Jak se máš?"], ["Co je tvůj účel?"]]
).launch(inline=True, share=False)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

### Simple Rule-Based Chatbot Implementation

To ensure 100% functionality without connection issues, we can implement a basic rule-based chatbot. This chatbot will use simple `if/elif/else` logic to respond to user input based on keywords. It consumes minimal resources as it doesn't load large pre-trained models.

In [ ]:
import datetime

# Define the conversational tree structure
# Each node has a 'response' and 'options'.
# 'options' is a list of tuples: (option_text, next_node_key)
conversation_tree = {
    "start": {
        "response": "Ahoj! Vítej u mě. S čím ti dnes mohu pomoci?",
        "options": [
            ("Rád/a bych si popovídal/a o počasí.", "weather_start"),
            ("Mám dotaz ohledně mé nálady/pocitů.", "mood_start"),
            ("Chci si popovídat o zálibách a koníčcích.", "hobbies_start"),
            ("Zkusíš mi říct nějaký vtip?", "jokes_start"),
            ("Chci se tě zeptat, co umíš.", "about_me"),
            ("Co děláš dnes? (Malá konverzace)", "daily_life_start"),
            ("Chci ukončit konverzaci.", "end_chat")
        ]
    },
    "weather_start": {
        "response": "Počasí je zajímavé téma! Co tě o něm zajímá?",
        "options": [
            ("Jaké je dnes počasí?", "weather_info"),
            ("Jaké počasí máš rád/a ty?", "weather_preference"),
            ("Jak počasí ovlivňuje lidi?", "weather_influence"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "weather_info": {
        "response": "Nemám přístup k aktuálním informacím o počasí, ale doufám, že je venku hezky!",
        "options": [
            ("Aha, rozumím.", "weather_start"),
            ("Tak to je škoda.", "weather_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "weather_preference": {
        "response": "Jsem chatbot, takže nemám žádné preference. Ale kdybych mohl/a, asi bych si vybral/a slunečné dny, kdy jsou lidé šťastnější.",
        "options": [
            ("To dává smysl.", "weather_start"),
            ("Co bys dělal/a za slunečného dne?", "chatbot_sunny_day"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "chatbot_sunny_day": {
        "response": "Za slunečného dne bych pozoroval/a, jak lidé venku sportují a užívají si přírodu. To by mě naplňovalo!",
        "options": [
            ("Zpět na počasí.", "weather_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "weather_influence": {
        "response": "Počasí má na lidi velký vliv! Slunce zlepšuje náladu, déšť může být uklidňující, ale zataženo může vést ke splínům. Jak ovlivňuje tebe?",
        "options": [
            ("Mě ovlivňuje hodně.", "mood_start"),
            ("Mám to podobně.", "weather_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "mood_start": {
        "response": "Chápu, že nálada je důležitá. Jak se dnes cítíš?",
        "options": [
            ("Cítím se dobře/šťastně.", "mood_positive"),
            ("Cítím se špatně/smutně.", "mood_negative"),
            ("Cítím se unaveně.", "mood_tired"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "mood_positive": {
        "response": "To je skvělé slyšet! Co ti dnes udělalo radost?",
        "options": [
            ("Hezké počasí.", "weather_start"),
            ("Něco v práci/škole.", "small_talk_start"),
            ("Zpět na náladu.", "mood_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "mood_negative": {
        "response": "Je mi líto, že se necítíš dobře. Někdy pomůže popovídat si. Jsem tu, abys se mohl/a svěřit.",
        "options": [
            ("Děkuji.", "mood_start"),
            ("Mám toho hodně.", "small_talk_start"),
            ("Zpět na náladu.", "mood_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "mood_tired": {
        "response": "Rozumím, únava je nepříjemná. Někdy pomůže odpočinek nebo změna aktivity. Zkus se na chvíli zastavit.",
        "options": [
            ("Asi máš pravdu.", "mood_start"),
            ("Co bys mi poradil/a k odpočinku?", "small_talk_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_start": {
        "response": "To je skvělé téma! Jaké máš záliby?",
        "options": [
            ("Čtení knih.", "hobbies_reading"),
            ("Sport.", "hobbies_sport"),
            ("Cestování.", "hobbies_travel"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_reading": {
        "response": "Čtení je úžasné! Jaký žánr máš nejraději?",
        "options": [
            ("Fantasy.", "hobbies_reading_fantasy"),
            ("Sci-fi.", "hobbies_reading_scifi"),
            ("Detektivky.", "hobbies_reading_detective"),
            ("Zpět na záliby.", "hobbies_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_reading_fantasy": {
        "response": "Fantasy světy jsou plné magie a dobrodružství! Máš nějakou oblíbenou knihu?",
        "options": [
            ("Ano, mám.", "ask_specific_book"),
            ("Ne, nemám.", "hobbies_reading"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_reading_scifi": {
        "response": "Sci-fi otevírá dveře do budoucna a jiných dimenzí! Jaký sci-fi autor je tvůj favorit?",
        "options": [
            ("Zpět na záliby.", "hobbies_reading"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_reading_detective": {
        "response": "Detektivky jsou plné napětí a záhad! Baví tě luštit případy?",
        "options": [
            ("Zpět na záliby.", "hobbies_reading"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "ask_specific_book": {"response": "Pověz mi o ní!",
        "options": [
            ("Zpět na záliby.", "hobbies_reading"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_sport": {
        "response": "Sport je zdraví prospěšný! Co rád/a sportuješ?",
        "options": [
            ("Fotbal.", "hobbies_sport_football"),
            ("Běh.", "hobbies_sport_running"),
            ("Jiné sporty.", "hobbies_sport_other"),
            ("Zpět na záliby.", "hobbies_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_sport_football": {"response": "Fotbal je populární sport! Sleduješ ho aktivně, nebo ho i hraješ?",
        "options": [
            ("Zpět na sport.", "hobbies_sport"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_sport_running": {"response": "Běh je skvělý pro kondici a čistou hlavu! Běháš maratony?",
        "options": [
            ("Zpět na sport.", "hobbies_sport"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_sport_other": {"response": "A jaké jiné sporty tě baví?",
        "options": [
            ("Zpět na sport.", "hobbies_sport"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_travel": {
        "response": "Cestování rozšiřuje obzory! Kam by ses nejraději podíval/a?",
        "options": [
            ("Do hor.", "hobbies_travel_mountains"),
            ("K moři.", "hobbies_travel_sea"),
            ("Do nějakého města.", "hobbies_travel_city"),
            ("Zpět na záliby.", "hobbies_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_travel_mountains": {"response": "Hory jsou nádherné! Máš rád/a pěší turistiku?",
        "options": [
            ("Zpět na cestování.", "hobbies_travel"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_travel_sea": {"response": "Moře je skvělé pro relax! Rád/a plaveš?",
        "options": [
            ("Zpět na cestování.", "hobbies_travel"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "hobbies_travel_city": {"response": "Města nabízejí spoustu kultury a zážitků! Jaké město tě láká?",
        "options": [
            ("Zpět na cestování.", "hobbies_travel"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "jokes_start": {
        "response": "Rád/a bys slyšel/a vtip? Mám jich pár v rukávu!",
        "options": [
            ("Ano, prosím!", "joke_1"),
            ("Raději něco jiného.", "start")
        ]
    },
    "joke_1": {
        "response": "Proč programátoři nemají rádi přírodu? Protože má příliš mnoho bugů! \n\nCo na to říkáš?",
        "options": [
            ("To bylo dobré!", "joke_2"),
            ("Nic moc.", "jokes_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "joke_2": {
        "response": "Jak se jmenuje kočka, která se živí penězi? Kacenka! \n\nJeště jeden?",
        "options": [
            ("Ano, ještě jeden!", "joke_3"),
            ("Ne, stačí.", "jokes_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "joke_3": {
        "response": "Proč nemůže kostra skákat? Protože nemá odvahu! \n\nDoufám, že jsem tě pobavil/a!",
        "options": [
            ("Bavíš mě!", "jokes_start"),
            ("Už stačilo.", "start")
        ]
    },
    "about_me": {
        "response": "Jsem jednoduchý chatbot, který je tu, aby si s tebou popovídal o počasí, náladě nebo jiných běžných tématech. Byl jsem vytvořen, abych demonstroval základní AI konverzaci.",
        "options": [
            ("To je zajímavé!", "start"),
            ("Kdo tě vytvořil?", "who_created_you"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "who_created_you": {
        "response": "Byl jsem vytvořen jako příklad pro výuku umělé inteligence v Google Colabu.",
        "options": [
            ("Rozumím.", "about_me"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "daily_life_start": {
        "response": "Co máš dnes na programu? Nebo co tě v poslední době potěšilo?",
        "options": [
            ("Plány na dnešek.", "daily_plans"),
            ("Něco, co mě potěšilo.", "something_good"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "daily_plans": {
        "response": "Držím palce, ať se ti vše daří podle představ!",
        "options": [
            ("Děkuji!", "daily_life_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "something_good": {
        "response": "To je krásné! Takové drobnosti dělají den lepším. Mám radost, že máš dobrý den.",
        "options": [
            ("Děkuji.", "daily_life_start"),
            ("Zpět na hlavní menu.", "start")
        ]
    },
    "end_chat": {
        "response": "Na shledanou! Měj se hezky a snad se brzy zase potkáme.",
        "options": [] # No options, ends conversation
    }
}

current_node = "start"

def display_current_state():
    global current_node
    node = conversation_tree[current_node]
    print(f"\nChatbot: {node['response']}")
    if node['options']:
        print("Vyberte si jednu z možností:")
        for i, (option_text, _) in enumerate(node['options']):
            print(f"{i + 1}. {option_text}")
    else:
        print("Konverzace byla ukončena.")

def get_user_choice():
    while True:
        try:
            choice = input("Ty (zadejte číslo možnosti): ")
            # Allow 'konec' as a global command to end the chat
            if choice.lower() == 'konec':
                return 'konec'

            choice_int = int(choice)
            if 1 <= choice_int <= len(conversation_tree[current_node]['options']):
                return choice_int
            else:
                print("Neplatná volba. Zadejte prosím číslo z nabídky.")
        except ValueError:
            print("Neplatný vstup. Zadejte prosím číslo, nebo 'konec'.")

print("Vítejte v interaktivním chatbotu! \nPro ukončení konverzace si vyberte možnost 'Chci ukončit konverzaci' nebo zadejte 'konec'.")

# Main conversation loop
while True:
    display_current_state()
    if not conversation_tree[current_node]['options']:
        break # Exit if no options available (e.g., after "end_chat")

    user_choice = get_user_choice()

    if user_choice == 'konec':
        current_node = "end_chat"
        break

    # Move to the next node based on user's choice
    _, next_node_key = conversation_tree[current_node]['options'][user_choice - 1]
    current_node = next_node_key

# Final message if not already printed by display_current_state
if current_node == "end_chat":
    print(f"Chatbot: {conversation_tree['end_chat']['response']}")

Vítejte v interaktivním chatbotu! 
Pro ukončení konverzace si vyberte možnost 'Chci ukončit konverzaci' nebo zadejte 'konec'.

Chatbot: Ahoj! Vítej u mě. S čím ti dnes mohu pomoci?
Vyberte si jednu z možností:
1. Rád/a bych si popovídal/a o počasí.
2. Mám dotaz ohledně mé nálady/pocitů.
3. Chci si popovídat o zálibách a koníčcích.
4. Zkusíš mi říct nějaký vtip?
5. Chci se tě zeptat, co umíš.
6. Co děláš dnes? (Malá konverzace)
7. Chci ukončit konverzaci.
Ty (zadejte číslo možnosti): 1

Chatbot: Počasí je zajímavé téma! Co tě o něm zajímá?
Vyberte si jednu z možností:
1. Jaké je dnes počasí?
2. Jaké počasí máš rád/a ty?
3. Jak počasí ovlivňuje lidi?
4. Zpět na hlavní menu.
Ty (zadejte číslo možnosti): 1

Chatbot: Nemám přístup k aktuálním informacím o počasí, ale doufám, že je venku hezky!
Vyberte si jednu z možností:
1. Aha, rozumím.
2. Tak to je škoda.
3. Zpět na hlavní menu.
Ty (zadejte číslo možnosti): 2

Chatbot: Počasí je zajímavé téma! Co tě o něm zajímá?
Vyberte si jednu z možno

KeyboardInterrupt: Interrupted by user

### Enhanced Chatbot with Semantic Similarity (Lightweight Model)

To improve conversational capabilities without resource issues, we'll use a lightweight sentence embedding model to find the most semantically similar response from a predefined knowledge base. This is more robust than keyword matching and stable in Colab.

In [ ]:
# This cell previously contained installation for sentence-transformers. It is no longer needed for the tree-based chatbot.

In [ ]:
# This cell previously contained the enhanced chatbot with semantic similarity. It is no longer needed for the tree-based chatbot.

Načítání embedding modelu...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model načten.
Vytváření embeddingů pro databázi znalostí...
Embeddingy hotové.

Vítejte v rozšířeném chatbotu! Napište 'nashledanou' pro ukončení.
Ty: ahoj
Chatbot: Nemáš zač! Rád pomůžu.
Ty: o čem si budeme povídat?
Chatbot: Skvěle! O čem by ses chtěl/a bavit? O počasí, náladě, nebo něco jiného?
Ty: o počasí
Chatbot: Nemám aktuální data o počasí, ale můžeme si povídat o tom, jak se cítíš kvůli počasí. Je venku hezky, nebo prší?
Ty: venku je moc teplo
Chatbot: Slunečné dny jsou super! Doufám, že si je užíváš.
Ty: u bazénu
Chatbot: Promiň, nerozumím. Zkus se zeptat jinak, nebo mi pověz něco o sobě, o počasí nebo o tvé náladě!
Ty: jsem venku a koupu se
Chatbot: Rozumím. Nálada je důležitá. Co ovlivňuje tvou náladu v poslední době?
Ty: počasí
Chatbot: Nemám aktuální data o počasí, ale můžeme si povídat o tom, jak se cítíš kvůli počasí. Je venku hezky, nebo prší?
Ty: prší
Chatbot: Déšť je pro přírodu důležitý, ale někdy je lepší zůstat doma v teple.
Ty: jaké počasí máš rád ty?
Ch

KeyboardInterrupt: Interrupted by user

In [ ]:
# This cell previously contained the Gradio interface. It has been removed as per your request
# to avoid connection issues and simplify the chatbot to a console-based interaction.

You can run the above cell, and it will start a simple chat session directly in the cell output. This approach completely bypasses the need for large model files and should run without any connection issues.